# Mobile Money Agent Network: Audit, Cleaning and Summary

**Analyst:** *Leah Atieno J.* | **Cohort:** *1MillionDevs * | **Date:** *24-11-26*

Engagement 2 of Assignment 3. Data: `agent_transactions.csv` and `agents.csv`, generated by `generate_data.py` (seed 44). All cleaning rules live in `cleaning_pipeline.py`; this notebook audits the raw data, calls `clean_agent_data()`, validates the merge and builds the summaries.

## 1. Setup and loading

In [ ]:
import os, subprocess, sys

# The raw CSVs are not committed. Create them with generate_data.py if they are missing.
RAW_FILES = ["agents.csv", "agent_transactions.csv"]
if not all(os.path.exists(f) for f in RAW_FILES):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import numpy as np
import pandas as pd

from cleaning_pipeline import clean_agent_data

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

# Second, text-only read so we can see literal strings such as "N/A" that pandas would silently turn into NaN
raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

print("transactions:", txns.shape, "| agents:", agents.shape)
txns.head()

transactions: (612, 5) | agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.00,2026-03-03
1,TX00002,AG020,deposit,"1,238.00",2026-04-27
2,TX00003,AG008,DEPOSIT,"9,129.00",2026-04-29
3,TX00004,AG009,deposit,652.00,2026-04-24
4,TX00005,AG009,withdrawal,478.00,2026-04-17


In [ ]:
txns.info()

<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB


## 2. Data-quality audit (Task A1)

Each count below is calculated in code from the raw file (612 rows, duplicates included), so no number is typed by hand. The six dimensions are completeness, validity, accuracy, consistency, uniqueness and timeliness.

In [ ]:
amount_num = pd.to_numeric(txns["amount"], errors="coerce")
q1, q3 = amount_num.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = q1 - 1.5 * iqr

dates = pd.to_datetime(txns["txn_date"].str.replace("/", "-"), format="%Y-%m-%d", errors="coerce")
onboard = txns["agent_id"].map(agents.set_index("agent_id")["onboarded"]).pipe(pd.to_datetime)

exact_dups = txns.duplicated().sum()
key_dups = txns.duplicated(subset="txn_id").sum()

audit = pd.DataFrame([
    ("Repeated txn_id (duplicate transactions)", "Uniqueness", "txn_id", key_dups),
    ("...of which differ from the original in another field", "Uniqueness", "agent_id", key_dups - exact_dups),
    ("txn_type not exactly 'Deposit' or 'Withdrawal' (case / trailing space)", "Consistency", "txn_type",
        (~txns["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("txn_date written as YYYY/MM/DD instead of YYYY-MM-DD", "Consistency", "txn_date",
        txns["txn_date"].str.contains("/").sum()),
    ("amount is empty (missing)", "Completeness", "amount", (raw["amount"] == "").sum()),
    ("amount holds the placeholder text 'N/A'", "Validity", "amount", (raw["amount"] == "N/A").sum()),
    ("agent_id not in the agent register (AG099)", "Validity", "agent_id",
        (~txns["agent_id"].isin(agents["agent_id"])).sum()),
    (f"amount above the IQR upper fence ({upper_fence:,.0f})", "Accuracy", "amount", (amount_num > upper_fence).sum()),
    ("...of which exactly 250,000 (about 9x the next largest value)", "Accuracy", "amount", (amount_num == 250000).sum()),
    ("amount below the IQR lower fence", "Accuracy", "amount", (amount_num < lower_fence).sum()),
    ("txn_date unparseable, in the future, or before the agent was onboarded", "Timeliness", "txn_date",
        (dates.isna() | (dates > pd.Timestamp.today()) | (dates < onboard)).sum()),
    ("Duplicate agent_id in the register", "Uniqueness", "agent_id (agents.csv)", agents["agent_id"].duplicated().sum()),
], columns=["issue", "dimension", "column", "rows affected"])
audit["rows affected"] = audit["rows affected"].astype(int)
audit

,issue,dimension,column,rows affected
0,Repeated txn_id (duplicate transactions),Uniqueness,txn_id,12
1,...of which differ from the original in anothe...,Uniqueness,agent_id,1
2,txn_type not exactly 'Deposit' or 'Withdrawal'...,Consistency,txn_type,384
3,txn_date written as YYYY/MM/DD instead of YYYY...,Consistency,txn_date,5
4,amount is empty (missing),Completeness,amount,15
5,amount holds the placeholder text 'N/A',Validity,amount,7
6,agent_id not in the agent register (AG099),Validity,agent_id,8
7,"amount above the IQR upper fence (6,044)",Accuracy,amount,40
8,"...of which exactly 250,000 (about 9x the next...",Accuracy,amount,5
9,amount below the IQR lower fence,Accuracy,amount,0


**Reading the audit.**

- Counts are raw-file rows, so a few overlap (for example a duplicated row that also has a bad date is counted in both lines) and they should not be added up.
- Pandas reads the text `N/A` as missing by default, so a plain `isna()` check hides it. Reading the file again as text shows **15 truly empty** and **7 `N/A`** amounts, which together make the 22 missing values `isna()` reports.
- Of the 12 repeated `txn_id` values, 11 are exact copies and **1 is not**: the second copy of one transaction carries the unknown agent `AG099` while the original has a valid agent. Dropping only exact duplicates (`drop_duplicates()` with no `subset`) would leave one `txn_id` twice, so de-duplication has to be by `txn_id`.
- **Timeliness: no issue found.** All dates fall between 2026-03-01 and 2026-04-29, none is in the future, and every transaction is after its agent's onboarding date. A zero is still a result worth recording.

## 3. Cleaning (Tasks A2 to A6 and A9)

One call to `clean_agent_data()` applies every rule in `cleaning_pipeline.py`. The decisions behind the rules:

| Task | Decision | Reason |
|---|---|---|
| A2 Uniqueness | Drop repeated `txn_id`, keep the **first** row | The first row is the original; the copy is appended at the end of the extract. Keying on `txn_id` also removes the near-duplicate. |
| A3 Consistency | Strip spaces, then title-case `txn_type` | Gives exactly `Deposit` and `Withdrawal`. Any other value would become missing and be logged (none occurs). |
| A4 Validity | `to_numeric(errors="coerce")` on amounts; `YYYY/MM/DD` fixed to `YYYY-MM-DD` before `to_datetime` | Bad text becomes missing instead of crashing, so nothing is lost silently. |
| A5 Completeness | **Flag, then fill** missing amounts with the median non-outlier amount for the same `txn_type`; `amount_imputed` marks the rows | Dropping them would delete valid transaction counts, dates and agents. The median is not pulled by the large outliers. Totals that include imputed rows are estimates, so the flag lets anyone exclude them. |
| A6 Outliers | Flag with the IQR rule in `is_outlier`; **do not delete** | An outlier is not automatically an error. The flag is set before imputation, on real values only. |
| A7 Merge | `how="left"`, `validate="many_to_one"`; unknown agents get county `Unknown` | An inner join would silently discard transactions and hide the problem. |

In [ ]:
clean, log = clean_agent_data(txns, agents)
log

,step,dimension,action,rows_affected,rows_after
0,0. Input,-,Raw transaction rows received,0,612
1,1. Duplicates,Uniqueness,"Dropped repeated txn_id, kept first occurrence...",12,600
2,2. txn_type,Consistency,Stripped spaces and standardised case to Depos...,376,600
3,3a. amount type,Validity,Converted amount to numeric; non-numeric text ...,0,600
4,3b. txn_date type,Consistency,Parsed dates to datetime (YYYY/MM/DD variants ...,5,600
5,4. Outliers,Accuracy,"Flagged amounts outside [-1,928, 6,084] (IQR r...",40,600
6,5. Missing amounts,Completeness,Flagged in amount_imputed; filled with median ...,20,600
7,6. Merge,Consistency,Left join to agent register (many_to_one); unk...,7,600


In [ ]:
print("Before:", len(txns), "rows   After:", len(clean), "rows   Removed:", len(txns) - len(clean))

assert clean["txn_id"].is_unique
assert set(clean["txn_type"]) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_float_dtype(clean["amount"]) and clean["amount"].notna().all()
assert pd.api.types.is_datetime64_any_dtype(clean["txn_date"])
print("All validation checks passed")
clean.dtypes

Before: 612 rows   After: 600 rows   Removed: 12
All validation checks passed


txn_id                       str
agent_id                     str
txn_type                  string
amount                   float64
txn_date          datetime64[us]
is_outlier                  bool
amount_imputed              bool
county                       str
agent_known                 bool
dtype: object

## 4. Outliers: the KES 250,000 values (Task A6)

In [ ]:
big = clean.loc[clean["amount"] == 250000, ["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]]
print("Rows flagged as IQR outliers:", clean["is_outlier"].sum())
print("...of which exactly 250,000:", len(big))
print("Largest flagged amount that is NOT 250,000:", f"{clean.loc[clean['is_outlier'] & (clean['amount'] != 250000), 'amount'].max():,.0f}")
print("Typical (median) normal amount:", f"{clean.loc[~clean['is_outlier'], 'amount'].median():,.0f}")
total, total_ex = clean["amount"].sum(), clean.loc[clean["amount"] != 250000, "amount"].sum()
print(f"Total value with / without the 250,000 rows: {total:,.0f} / {total_ex:,.0f}  (they are {(total - total_ex) / total * 100:.1f}% of all value)")
big

Rows flagged as IQR outliers: 40
...of which exactly 250,000: 5
Largest flagged amount that is NOT 250,000: 28,847
Typical (median) normal amount: 1,704
Total value with / without the 250,000 rows: 2,686,740 / 1,436,740  (they are 46.5% of all value)


,txn_id,agent_id,county,txn_type,amount,txn_date
306,TX00307,AG012,Kisumu,Deposit,"250,000.00",2026-03-02
315,TX00316,AG014,Nairobi,Deposit,"250,000.00",2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,"250,000.00",2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,"250,000.00",2026-04-11
594,TX00595,AG016,Kiambu,Deposit,"250,000.00",2026-03-11


**Findings.** The IQR rule flags **40** transactions above KES 6,044. Most (35) are the natural long tail of transaction sizes: the largest of these is KES 28,847, about 17 times the median of KES 1,704 and a plausible tail for a skewed distribution. The **5 transactions of exactly KES 250,000** are different: they are almost nine times larger than the next largest value, share one identical round amount, and together make **46.5%** of the total transaction value from just 5 of 600 rows.

**Recommendation.** Do not delete them and do not accept them as genuine yet. (1) Keep them in the data with `is_outlier = True`. (2) Send the five `txn_id`s to the provider's operations or fraud team to check against the source system, because an identical round value can indicate a default, a manual entry or a system limit being reached (the cause is not something the data can tell us). (3) Until confirmed, report county totals **both with and without** these rows, as done below, and use medians rather than means for typical size. (4) Add a validation rule at the source for amounts above the agreed limit.

## 5. Merge validation (Task A7)

In [ ]:
# Independent check on the de-duplicated keys, using indicator=True
check = (txns.drop_duplicates("txn_id")
              .merge(agents, on="agent_id", how="left", indicator=True, validate="many_to_one"))

print("Transactions before merge (after de-duplication):", txns["txn_id"].nunique())
print("Rows after left join                              :", len(check))
print("Rows in clean_df                                  :", len(clean))
print()
print(check["_merge"].value_counts().to_string())
print()
unknown = clean.loc[~clean["agent_known"]]
print("Transactions with unknown agents:", len(unknown), "| agent ids:", unknown["agent_id"].unique().tolist(),
      "| value KES", f"{unknown['amount'].sum():,.0f}")
print("An inner join would have dropped", len(unknown), "rows:", len(check) - (check["_merge"] == "both").sum())
print("Agents in the register with no transactions:", (~agents["agent_id"].isin(clean["agent_id"])).sum())

Transactions before merge (after de-duplication): 600
Rows after left join                              : 600
Rows in clean_df                                  : 600

_merge
both          593
left_only       7
right_only      0

Transactions with unknown agents: 7 | agent ids: ['AG099'] | value KES 27,156
An inner join would have dropped 7 rows: 7
Agents in the register with no transactions: 0


**Result.** The left join keeps all 600 rows. **7 transactions** reference agent `AG099`, which is not in the register (8 raw rows, one of which was removed as a duplicate). I kept them with county `Unknown` rather than dropping them, because the money moved and the transactions are likely real; the likely causes are a missing register entry (an agent onboarded but never registered) or a typo in the legacy system. They should go to the data owner for correction. The `Unknown` row stays visible in the summaries so county totals add up to the full total.

## 6. Summaries (Task A8)

### (a) County summary, split by transaction type

In [ ]:
county = (clean.groupby(["county", "txn_type"])
               .agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean"))
               .reset_index()
               .sort_values(["county", "txn_type"]))
county

,county,txn_type,total_value,n_transactions,avg_value
0,Kiambu,Deposit,"334,737.00",43,"7,784.58"
1,Kiambu,Withdrawal,"76,920.00",27,"2,848.89"
2,Kisumu,Deposit,"436,242.00",74,"5,895.16"
3,Kisumu,Withdrawal,"76,244.00",37,"2,060.65"
4,Mombasa,Deposit,"148,748.00",72,"2,065.94"
5,Mombasa,Withdrawal,"368,592.00",61,"6,042.49"
6,Nairobi,Deposit,"543,352.00",118,"4,604.68"
7,Nairobi,Withdrawal,"161,514.00",62,"2,605.06"
8,Nakuru,Deposit,"184,771.00",60,"3,079.52"
9,Nakuru,Withdrawal,"328,464.00",39,"8,422.15"


In [ ]:
county_total = (clean.groupby("county")
                     .agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean"),
                          median_value=("amount", "median"))
                     .sort_values("total_value", ascending=False))
county_total["share_of_value_pct"] = county_total["total_value"] / county_total["total_value"].sum() * 100

# Same view without the five 250,000 rows, to show their influence
ex = clean[clean["amount"] != 250000]
county_total["total_value_ex_250k"] = ex.groupby("county")["amount"].sum()
county_total["total_value_ex_250k"] = county_total["total_value_ex_250k"].fillna(0)
county_total.loc["All counties"] = [county_total["total_value"].sum(), county_total["n_transactions"].sum(),
                                   clean["amount"].mean(), clean["amount"].median(), 100.0,
                                   county_total["total_value_ex_250k"].sum()]
county_total

,total_value,n_transactions,avg_value,median_value,share_of_value_pct,total_value_ex_250k
county,,,,,,
Nairobi,"704,866.00",180.00,"3,915.92","1,905.00",26.23,"454,866.00"
Mombasa,"517,340.00",133.00,"3,889.77","1,595.00",19.26,"267,340.00"
Nakuru,"513,235.00",99.00,"5,184.19","1,880.00",19.10,"263,235.00"
Kisumu,"512,486.00",111.00,"4,616.99","1,674.00",19.07,"262,486.00"
Kiambu,"411,657.00",70.00,"5,880.81","1,825.50",15.32,"161,657.00"
Unknown,"27,156.00",7.00,"3,879.43","3,110.00",1.01,"27,156.00"
All counties,"2,686,740.00",600.00,"4,477.90","1,745.50",100.00,"1,436,740.00"


### (b) Weekly transaction value by county

In [ ]:
weekly = clean.assign(week_starting=clean["txn_date"].dt.to_period("W-SUN").dt.start_time.dt.strftime("%Y-%m-%d"))
weekly_pivot = weekly.pivot_table(index="week_starting", columns="county", values="amount",
                                  aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_starting,,,,,,,
2026-02-23,0.00,"1,893.00","6,759.00","7,046.00","2,684.00",0.00,"18,382.00"
2026-03-02,"18,317.00","287,933.00","285,610.00","68,967.00","56,574.00",0.00,"717,401.00"
2026-03-09,"267,641.00","23,186.00","30,418.00","52,801.00","28,730.00",0.00,"402,776.00"
2026-03-16,"22,054.00","27,935.00","25,085.00","43,048.00","39,795.00","1,639.00","159,556.00"
2026-03-23,"11,504.00","27,077.00","17,050.00","41,314.00","18,868.00",0.00,"115,813.00"
2026-03-30,"22,609.00","41,077.00","38,344.00","282,931.00","18,145.00","8,507.00","411,613.00"
2026-04-06,"15,030.00","15,341.00","47,033.00","56,727.00","276,609.00","9,824.00","420,564.00"
2026-04-13,"12,265.00","29,297.00","26,730.00","56,302.00","26,093.00",0.00,"150,687.00"
2026-04-20,"24,507.00","37,439.00","16,687.00","64,926.00","30,862.00","1,877.00","176,298.00"


In [ ]:
assert abs(weekly_pivot.loc["Total", "Total"] - clean["amount"].sum()) < 1e-6
assert abs(county_total.loc["All counties", "total_value"] - clean["amount"].sum()) < 1e-6
print("Pivot and county totals both reconcile to", f"KES {clean['amount'].sum():,.0f}")

Pivot and county totals both reconcile to KES 2,686,740


In [ ]:
top = county_total.drop("All counties").index[0]
print("County with the highest total value:", top)
print("County with most transactions     :", county_total.drop("All counties")["n_transactions"].idxmax())
print("Highest value excluding 250,000 rows:", county_total.drop("All counties")["total_value_ex_250k"].idxmax())

County with the highest total value: Nairobi
County with most transactions     : Nairobi
Highest value excluding 250,000 rows: Nairobi


**Reading the tables.** Value and transaction counts are driven partly by how many agents a county has (the register assigns counties to agents at random, so some counties have many agents and some few), so county totals say little about agent performance. To compare agents fairly, divide by the number of agents or look at value per agent. Counties whose total changes a lot when the five 250,000 rows are removed (compare `total_value` with `total_value_ex_250k`) should not be ranked until those transactions are verified. Imputed amounts (see `amount_imputed`) account for a small part of each total and are estimates.

## 7. Export

In [ ]:
os.makedirs("outputs", exist_ok=True)
clean.to_csv("outputs/agent_transactions_clean.csv", index=False, date_format="%Y-%m-%d")

out = pd.read_csv("outputs/agent_transactions_clean.csv")
print("Saved", out.shape)
print("Duplicate txn_id:", out["txn_id"].duplicated().sum(), "| txn_type:", sorted(out["txn_type"].unique()))
print("txn_date sample:", out["txn_date"].head(3).tolist(), "| all ISO:", out["txn_date"].str.match(r"^\d{4}-\d{2}-\d{2}$").all())
print("Columns:", out.columns.tolist())

Saved (600, 9)
Duplicate txn_id: 0 | txn_type: ['Deposit', 'Withdrawal']
txn_date sample: ['2026-03-03', '2026-04-27', '2026-04-29'] | all ISO: True
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'amount_imputed', 'county', 'agent_known']


## 8. Data Quality Report

**Scope.** Mobile-money transaction extract from the legacy system (612 rows, 5 columns, 2026-03-01 to 2026-04-29) and the agent register (20 agents). After cleaning the extract has **600 rows** (612 before, 12 removed) and 9 columns. Every rule is in `cleaning_pipeline.py` and was applied by code; no row was edited by hand.

| Dimension | Issue found | Rows | Treatment | Residual risk |
|---|---|---|---|---|
| **Completeness** | Missing amounts (15 empty, 7 written as `N/A`) | 20 after de-duplication | Flagged in `amount_imputed`, filled with the median non-outlier amount for the same transaction type | Totals that include imputed rows are estimates. Ask the source system for the real amounts. |
| **Validity** | `N/A` text in a numeric field; agent `AG099` not in the register | 7 text; 7 unknown-agent transactions | Text converted to missing, then handled as above; unknown agents kept with county `Unknown` | The `Unknown` county cannot be attributed to a real area until the register is corrected. |
| **Accuracy** | 40 amounts above the IQR fence, including 5 identical values of KES 250,000 | 40 | Flagged in `is_outlier`, kept in the data, reported with and without the 250,000 rows | If the 250,000 rows are errors, they overstate value by 46.5%. Verify with the source before any ranking. |
| **Consistency** | `txn_type` written in 5 variants (case and trailing spaces); dates in two formats (`YYYY-MM-DD` and `YYYY/MM/DD`) | 384 types; 5 dates | Standardised to `Deposit`/`Withdrawal` and ISO dates | None after cleaning. A validation rule at the source would stop it returning. |
| **Uniqueness** | Repeated `txn_id` values (11 exact copies and 1 copy with a different agent) | 12 | Kept the first occurrence of each `txn_id` | Keeping the first row assumes the original comes first, which holds here because copies sit at the end of the extract. |
| **Timeliness** | Checked for future dates, unparseable dates and transactions before the agent's onboarding date | 0 | None needed | The extract covers only two months, so seasonality cannot be judged. |

**Overall judgement.** After cleaning, the extract is fit for summarising counts and typical transaction sizes by county and week. It is **not yet fit for ranking counties or agents by total value**, because of the five unverified KES 250,000 transactions, the 20 imputed amounts and the 7 unattributed transactions. The three recommended actions are: (1) verify the five 250,000 transactions, (2) correct or add `AG099` in the agent register, (3) add validation at the source for `txn_type`, `amount` and date format.

**Privacy (Task A10).** The current extract holds no names or phone numbers, but if customer **phone numbers** were included they would be **personal data** under the Kenya Data Protection Act, 2019, because a phone number identifies a person directly, and so would customer names and national ID numbers. Fields that identify people indirectly deserve care too: a transaction date and amount can single out one customer at a small agent, and agent identifiers may themselves be personal data when an agent is an individual trader. I would protect this data by (1) **minimising**: exclude phone numbers from analysis extracts unless the purpose requires them, in line with the Act's principles of purpose limitation and data minimisation; (2) **pseudonymising**: replace each phone number with a keyed hash or token, keeping the key and the lookup table separately under restricted access; (3) **masking** any number that must be displayed (for example `07** *** 678`); (4) **access control and encryption**: store the extract encrypted, give access only to named analysts, and log that access; (5) **aggregating**: publish county and weekly totals, and suppress cells with very few customers; (6) **retention**: delete raw extracts when the work ends; and (7) accountability: record the lawful basis, check that the provider is registered with the Office of the Data Protection Commissioner, and have a process for data-subject requests and breach reporting. I am not a lawyer, so the data-protection officer should confirm these steps.